# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [8]:
import re
from collections import Counter
from math import log

import nltk
import pandas as pd
from nltk.corpus import gutenberg

nltk.download("gutenberg", quiet=True)


def preprocesar(texto):
    return re.findall(r"[^\W\d_]+", texto.lower())


filas = []

for doc_id in gutenberg.fileids():
    texto = gutenberg.raw(doc_id)
    encabezado = re.search(r"\[([^\]]+)\]", texto[:500])
    titulo = " ".join(encabezado.group(1).split()) if encabezado else doc_id

    filas.append({
        "Identificador de documento": str(doc_id),
        "Título": titulo,
        "Texto": texto,
        "tokens": preprocesar(texto),
    })

corpus = pd.DataFrame(filas)

N = len(corpus)
df_corpus = Counter()

for tokens in corpus["tokens"]:
    df_corpus.update(set(tokens))

idf_corpus = {
    termino: log(N / df) + 1
    for termino, df in df_corpus.items()
}

estadisticas_terminos = pd.DataFrame([
    {"Término": termino, "DF": df, "IDF": idf_corpus[termino]}
    for termino, df in df_corpus.items()
])

print(f"Documentos: {N} | Términos distintos: {len(df_corpus)}")
display(corpus.head())
display(estadisticas_terminos.head())

Documentos: 18 | Términos distintos: 41509


,Identificador de documento,Título,Texto,tokens
0,austen-emma.txt,Emma by Jane Austen 1816,[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAP...,"[emma, by, jane, austen, volume, i, chapter, i..."
1,austen-persuasion.txt,Persuasion by Jane Austen 1818,[Persuasion by Jane Austen 1818]\n\n\nChapter ...,"[persuasion, by, jane, austen, chapter, sir, w..."
2,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,[Sense and Sensibility by Jane Austen 1811]\n\...,"[sense, and, sensibility, by, jane, austen, ch..."
3,bible-kjv.txt,The King James Bible,[The King James Bible]\n\nThe Old Testament of...,"[the, king, james, bible, the, old, testament,..."
4,blake-poems.txt,Poems by William Blake 1789,[Poems by William Blake 1789]\n\n \nSONGS OF I...,"[poems, by, william, blake, songs, of, innocen..."


,Término,DF,IDF
0,absurd,10,1.587787
1,song,16,1.117783
2,untouched,5,2.280934
3,respectability,6,2.098612
4,playfully,3,2.791759


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [9]:
import pandas as pd
from IPython.display import display

consultas = [
    ("the", "Comprobar qué recupera una consulta con un artículo muy frecuente."),
    ("love", "Encontrar libros que hablan del amor."),
    ("war peace", "Encontrar textos sobre la guerra y la paz."),
    ("queequeg", "Encontrar el libro que menciona al personaje Queequeg."),
    ("elinor dashwood", "Localizar textos en los que se menciona a Elinor Dashwood."),
    ("whale ship captain", "Encontrar relatos sobre barcos, capitanes y ballenas."),
    ("king queen royal court", "Encontrar pasajes sobre la realeza y la corte."),
    ("faith prayer god heaven salvation", "Encontrar textos sobre la fe y la salvación."),
]

N = len(corpus)

tabla_consultas = pd.DataFrame([
    {
        "Consulta": consulta,
        "Qué busca el usuario": objetivo,
        "Longitud": len(preprocesar(consulta)),
        "DF por término": ", ".join(
            f"{termino}: {df_corpus[termino]}/{N}"
            for termino in preprocesar(consulta)
        ),
    }
    for consulta, objetivo in consultas
])

display(tabla_consultas)

,Consulta,Qué busca el usuario,Longitud,DF por término
0,the,Comprobar qué recupera una consulta con un art...,1,the: 18/18
1,love,Encontrar libros que hablan del amor.,1,love: 15/18
2,war peace,Encontrar textos sobre la guerra y la paz.,2,"war: 10/18, peace: 16/18"
3,queequeg,Encontrar el libro que menciona al personaje Q...,1,queequeg: 1/18
4,elinor dashwood,Localizar textos en los que se menciona a Elin...,2,"elinor: 1/18, dashwood: 1/18"
5,whale ship captain,"Encontrar relatos sobre barcos, capitanes y ba...",3,"whale: 6/18, ship: 13/18, captain: 9/18"
6,king queen royal court,Encontrar pasajes sobre la realeza y la corte.,4,"king: 15/18, queen: 15/18, royal: 9/18, court:..."
7,faith prayer god heaven salvation,Encontrar textos sobre la fe y la salvación.,5,"faith: 15/18, prayer: 10/18, god: 16/18, heave..."


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [14]:
from itertools import combinations, zip_longest
from IPython.display import display

# Busca el título y los tokens de cada documento por su ID.
titulos_por_id = dict(zip(
    corpus["Identificador de documento"].astype(str), corpus["Título"]
))
tokens_por_id = dict(zip(
    corpus["Identificador de documento"].astype(str), corpus["tokens"]
))


def pares_id_score(ranking):
    return [
        (str(doc["doc_id"]), doc["Score"]) if isinstance(doc, dict)
        else (str(doc[0]), doc[1])
        for doc in ranking
    ]


def mostrar_documento(resultado):
    if resultado is None:
        return "—", "—"

    doc_id, score = resultado
    return f"{doc_id} | {titulos_por_id[doc_id]}", f"{score:.6f}"
comparaciones = {}

for entrada in consultas:
    consulta = entrada if isinstance(entrada, str) else entrada[0]

    tfidf, bm25 = comparar_consulta(consulta)
    tfidf = pares_id_score(tfidf)
    bm25 = pares_id_score(bm25)
    comparaciones[consulta] = (tfidf, bm25)

    filas = []
    for posicion, (a, b_doc) in enumerate(zip_longest(tfidf, bm25), start=1):
        nombre_a, coseno = mostrar_documento(a)
        nombre_b, score_bm25 = mostrar_documento(b_doc)
        filas.append([posicion, nombre_a, coseno, nombre_b, score_bm25])

    print(f"\nConsulta: {consulta}")
    display(pd.DataFrame(filas, columns=[
        "Posición", "ID y título TF-IDF", "Coseno", "ID y título BM25", "Score BM25"
    ]))

    ids_compartidos = set(dict(tfidf)) & set(dict(bm25))
    primero_tfidf, _ = mostrar_documento(next(iter(tfidf), None))
    primero_bm25, _ = mostrar_documento(next(iter(bm25), None))

    print(f"IDs compartidos: {len(ids_compartidos)}")
    print(f"Tamaño real: TF-IDF = {len(tfidf)} | BM25 = {len(bm25)}")
    print(f"Primer documento TF-IDF: {primero_tfidf}")
    print(f"Primer documento BM25: {primero_bm25}")
ejemplo = None

for consulta in sorted(comparaciones, key=lambda q: -len(preprocesar(q))):
    tfidf, bm25 = comparaciones[consulta]
    pos_tfidf = {d: i for i, (d, _) in enumerate(tfidf, start=1)}
    pos_bm25 = {d: i for i, (d, _) in enumerate(bm25, start=1)}
    comunes = [d for d in pos_tfidf if d in pos_bm25]

    for doc_a, doc_b in combinations(comunes, 2):
        distancia_tfidf = abs(pos_tfidf[doc_a] - pos_tfidf[doc_b])
        distancia_bm25 = abs(pos_bm25[doc_a] - pos_bm25[doc_b])
        cambian_ambos = all(
            pos_tfidf[d] != pos_bm25[d] for d in (doc_a, doc_b)
        )

        if cambian_ambos or distancia_tfidf != distancia_bm25:
            ejemplo = (doc_a, doc_b)
            break

    if ejemplo is not None:
        break
if ejemplo is None:
    print("\nLas consultas no produjeron un par compartido con el cambio solicitado.")
else:
    scores_tfidf = dict(tfidf)
    scores_bm25 = dict(bm25)
    terminos = list(dict.fromkeys(preprocesar(consulta)))

    print(f"\nConsulta elegida: {consulta}")
    print(f"Distancia entre posiciones: TF-IDF = {distancia_tfidf} | BM25 = {distancia_bm25}")
    print(f"Longitud media del corpus: {longitud_media:,.1f} tokens")

    display(pd.DataFrame([
        {
            "doc_id": d,
            "Posición TF-IDF": pos_tfidf[d],
            "Coseno": scores_tfidf[d],
            "Posición BM25": pos_bm25[d],
            "Score BM25": scores_bm25[d],
        }
        for d in ejemplo
    ]))

    filas = []
    aportes = {}

    for d in ejemplo:
        tokens = tokens_por_id[d]
        longitud = len(tokens)
        ajuste = k1 * (1 - b + b * longitud / longitud_media)

        for t in terminos:
            tf = tokens.count(t)
            aporte = idf_bm25.get(t, 0) * tf * (k1 + 1) / (tf + ajuste)
            aportes[d, t] = aporte

            filas.append({
                "doc_id": d,
                "|d|": longitud,
                "Término": t,
                "f(t,d)": tf,
                "df(t)": df_corpus.get(t, 0),
                "Aporte BM25": aporte,
            })

    display(pd.DataFrame(filas))

    for metodo, posiciones, scores in [
        ("TF-IDF + coseno", pos_tfidf, scores_tfidf),
        ("BM25", pos_bm25, scores_bm25),
    ]:
        primero, segundo = sorted(ejemplo, key=posiciones.get)
        motivo = (
            "desempate por ID" if scores[primero] == scores[segundo]
            else "mayor puntuación"
        )
        print(
            f"{metodo}: {primero} ({scores[primero]:.6f}) queda antes que "
            f"{segundo} ({scores[segundo]:.6f}) por {motivo}."
        )

    termino = max(terminos, key=lambda t: abs(aportes[doc_a, t] - aportes[doc_b, t]))
    print(f"\nTérmino con mayor diferencia de aporte: {termino!r}")
    print(f"DF: {df_corpus.get(termino, 0)} de {len(corpus)} documentos.")

    for d in ejemplo:
        tokens = tokens_por_id[d]
        print(
            f"{d}: longitud = {len(tokens):,}, "
            f"frecuencia = {tokens.count(termino):,}, "
            f"aporte = {aportes[d, termino]:.6f}."
        )


Consulta: the


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,whitman-leaves.txt | Leaves of Grass by Walt W...,0.723446,bible-kjv.txt | The King James Bible,0.058664
1,2,melville-moby_dick.txt | Moby Dick by Herman M...,0.669970,whitman-leaves.txt | Leaves of Grass by Walt W...,0.058663
2,3,bryant-stories.txt | Stories to Tell to Childr...,0.666026,melville-moby_dick.txt | Moby Dick by Herman M...,0.058662
3,4,chesterton-brown.txt | The Wisdom of Father Br...,0.633625,chesterton-brown.txt | The Wisdom of Father Br...,0.058659
4,5,bible-kjv.txt | The King James Bible,0.619686,chesterton-ball.txt | The Ball and The Cross b...,0.058659
5,6,blake-poems.txt | Poems by William Blake 1789,0.611702,bryant-stories.txt | Stories to Tell to Childr...,0.058659
6,7,chesterton-thursday.txt | The Man Who Was Thur...,0.588002,chesterton-thursday.txt | The Man Who Was Thur...,0.058658
7,8,chesterton-ball.txt | The Ball and The Cross b...,0.582681,edgeworth-parents.txt | The Parent's Assistant...,0.058658
8,9,carroll-alice.txt | Alice's Adventures in Wond...,0.547424,austen-persuasion.txt | Persuasion by Jane Aus...,0.058654
9,10,edgeworth-parents.txt | The Parent's Assistant...,0.495650,austen-emma.txt | Emma by Jane Austen 1816,0.058653


IDs compartidos: 8
Tamaño real: TF-IDF = 10 | BM25 = 10
Primer documento TF-IDF: whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855
Primer documento BM25: bible-kjv.txt | The King James Bible

Consulta: love


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,blake-poems.txt | Poems by William Blake 1789,0.047776,whitman-leaves.txt | Leaves of Grass by Walt W...,0.445878
1,2,whitman-leaves.txt | Leaves of Grass by Walt W...,0.023259,milton-paradise.txt | Paradise Lost by John Mi...,0.444030
2,3,milton-paradise.txt | Paradise Lost by John Mi...,0.018184,blake-poems.txt | Poems by William Blake 1789,0.442537
3,4,austen-emma.txt | Emma by Jane Austen 1816,0.009013,austen-emma.txt | Emma by Jane Austen 1816,0.442142
4,5,austen-sense.txt | Sense and Sensibility by Ja...,0.007895,austen-sense.txt | Sense and Sensibility by Ja...,0.440956
5,6,edgeworth-parents.txt | The Parent's Assistant...,0.006294,edgeworth-parents.txt | The Parent's Assistant...,0.439471
6,7,austen-persuasion.txt | Persuasion by Jane Aus...,0.006158,bible-kjv.txt | The King James Bible,0.439017
7,8,bible-kjv.txt | The King James Bible,0.003559,austen-persuasion.txt | Persuasion by Jane Aus...,0.438133
8,9,chesterton-ball.txt | The Ball and The Cross b...,0.003458,chesterton-ball.txt | The Ball and The Cross b...,0.431889
9,10,bryant-stories.txt | Stories to Tell to Childr...,0.002966,bryant-stories.txt | Stories to Tell to Childr...,0.426459


IDs compartidos: 10
Tamaño real: TF-IDF = 10 | BM25 = 10
Primer documento TF-IDF: blake-poems.txt | Poems by William Blake 1789
Primer documento BM25: whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855

Consulta: war peace


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,milton-paradise.txt | Paradise Lost by John Mi...,0.015129,whitman-leaves.txt | Leaves of Grass by Walt W...,1.593044
1,2,whitman-leaves.txt | Leaves of Grass by Walt W...,0.012375,milton-paradise.txt | Paradise Lost by John Mi...,1.588216
2,3,blake-poems.txt | Poems by William Blake 1789,0.008069,bible-kjv.txt | The King James Bible,1.575050
3,4,bible-kjv.txt | The King James Bible,0.005499,chesterton-ball.txt | The Ball and The Cross b...,1.522148
4,5,shakespeare-caesar.txt | The Tragedie of Juliu...,0.005205,bryant-stories.txt | Stories to Tell to Childr...,1.516697
5,6,shakespeare-macbeth.txt | The Tragedie of Macb...,0.004187,chesterton-thursday.txt | The Man Who Was Thur...,1.481025
6,7,bryant-stories.txt | Stories to Tell to Childr...,0.003877,melville-moby_dick.txt | Moby Dick by Herman M...,1.476953
7,8,chesterton-ball.txt | The Ball and The Cross b...,0.003411,austen-persuasion.txt | Persuasion by Jane Aus...,1.465350
8,9,chesterton-thursday.txt | The Man Who Was Thur...,0.002514,chesterton-brown.txt | The Wisdom of Father Br...,1.355543
9,10,shakespeare-hamlet.txt | The Tragedie of Hamle...,0.002332,austen-emma.txt | Emma by Jane Austen 1816,0.791337


... (10 outputs left out)

IDs compartidos: 7
Tamaño real: TF-IDF = 10 | BM25 = 10
Primer documento TF-IDF: milton-paradise.txt | Paradise Lost by John Milton 1667
Primer documento BM25: bible-kjv.txt | The King James Bible

Consulta elegida: faith prayer god heaven salvation
Distancia entre posiciones: TF-IDF = 1 | BM25 = 1
Longitud media del corpus: 118,670.8 tokens


,doc_id,Posición TF-IDF,Coseno,Posición BM25,Score BM25
0,milton-paradise.txt,1,0.046565,2,5.059924
1,bible-kjv.txt,2,0.022009,1,5.775375


,doc_id,|d|,Término,"f(t,d)",df(t),Aporte BM25
0,milton-paradise.txt,80493,faith,35,15,0.436561
1,milton-paradise.txt,80493,prayer,6,10,1.132839
2,milton-paradise.txt,80493,god,260,16,0.309290
3,milton-paradise.txt,80493,heaven,419,13,0.750218
4,milton-paradise.txt,80493,salvation,3,4,2.431015
5,bible-kjv.txt,791842,faith,247,15,0.436768
6,bible-kjv.txt,791842,prayer,109,10,1.233392
7,bible-kjv.txt,791842,god,4472,16,0.309936
8,bible-kjv.txt,791842,heaven,583,13,0.743804
9,bible-kjv.txt,791842,salvation,164,4,3.051475


TF-IDF + coseno: milton-paradise.txt (0.046565) queda antes que bible-kjv.txt (0.022009) por mayor puntuación.
BM25: bible-kjv.txt (5.775375) queda antes que milton-paradise.txt (5.059924) por mayor puntuación.

Término con mayor diferencia de aporte: 'salvation'
DF: 4 de 18 documentos.
milton-paradise.txt: longitud = 80,493, frecuencia = 3, aporte = 2.431015.
bible-kjv.txt: longitud = 791,842, frecuencia = 164, aporte = 3.051475.


**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [16]:
from collections import Counter

import pandas as pd
from IPython.display import display

consulta = "whale ship captain"
terminos = list(dict.fromkeys(preprocesar(consulta)))

configuraciones = [
    ("Base", 1.2, 0.75),
    ("Sin ajuste por longitud", 1.2, 0.00),
    ("Mayor efecto de las repeticiones", 2.0, 0.75),
]
frecuencias = {}
longitudes = {}

for doc_id, tokens in tokens_por_id.items():
    conteo = Counter(tokens)
    if any(conteo[t] > 0 for t in terminos):
        frecuencias[doc_id] = conteo
        longitudes[doc_id] = len(tokens)
resultados = {}

for nombre, k1_prueba, b_prueba in configuraciones:
    aportes = {}
    scores = {}

    for doc_id, conteo in frecuencias.items():
        longitud = longitudes[doc_id]
        ajuste = k1_prueba * (1 - b_prueba + b_prueba * longitud / longitud_media)

        aportes[doc_id] = {}
        for t in terminos:
            tf = conteo[t]
            aportes[doc_id][t] = (
                idf_bm25.get(t, 0) * tf * (k1_prueba + 1) / (tf + ajuste)
            )

        scores[doc_id] = sum(aportes[doc_id].values())

    orden = sorted(scores, key=lambda doc_id: (-scores[doc_id], doc_id))
    posiciones = {doc_id: pos for pos, doc_id in enumerate(orden, start=1)}
    resultados[nombre] = (posiciones, scores, aportes)
mas_corto = min(longitudes, key=longitudes.get)
mas_largo = max(longitudes, key=longitudes.get)
primero_base = min(resultados["Base"][0], key=resultados["Base"][0].get)
elegidos = list(dict.fromkeys([mas_corto, mas_largo, primero_base]))
filas_posiciones = []

for doc_id in elegidos:
    fila = {"doc_id": doc_id, "|d|": longitudes[doc_id]}

    for nombre, (posiciones, scores, _) in resultados.items():
        fila[f"Posición · {nombre}"] = posiciones[doc_id]
        fila[f"Score · {nombre}"] = scores[doc_id]

    filas_posiciones.append(fila)

print(f"Consulta: {consulta}")
print(f"Longitud media de la colección: {longitud_media:,.1f} tokens")
display(pd.DataFrame(filas_posiciones).round(6))
filas_aportes = []

for doc_id in elegidos:
    for t in terminos:
        fila = {
            "doc_id": doc_id,
            "Término": t,
            "f(t,d)": frecuencias[doc_id][t],
            "df(t)": df_corpus.get(t, 0),
        }

        for nombre, (_, _, aportes) in resultados.items():
            fila[f"Aporte · {nombre}"] = aportes[doc_id][t]

        filas_aportes.append(fila)

display(pd.DataFrame(filas_aportes).round(6))

Consulta: whale ship captain
Longitud media de la colección: 118,670.8 tokens


,doc_id,|d|,Posición · Base,Score · Base,Posición · Sin ajuste por longitud,Score · Sin ajuste por longitud,Posición · Mayor efecto de las repeticiones,Score · Mayor efecto de las repeticiones
0,shakespeare-macbeth.txt,18272,12,0.522634,13,0.341749,12,0.592301
1,bible-kjv.txt,791842,4,2.910066,3,3.936796,4,3.540963
2,melville-moby_dick.txt,218382,1,4.620972,1,4.626986,1,6.287248


,doc_id,Término,"f(t,d)",df(t),Aporte · Base,Aporte · Sin ajuste por longitud,Aporte · Mayor efecto de las repeticiones
0,shakespeare-macbeth.txt,whale,0,6,0.000000,0.000000,0.000000
1,shakespeare-macbeth.txt,ship,1,13,0.522634,0.341749,0.592301
2,shakespeare-macbeth.txt,captain,0,9,0.000000,0.000000,0.000000
3,bible-kjv.txt,whale,3,6,0.760790,1.685572,0.714621
4,bible-kjv.txt,ship,71,13,0.690525,0.739352,0.893063
5,bible-kjv.txt,captain,139,9,1.458752,1.511872,1.933279
6,melville-moby_dick.txt,whale,1226,6,2.356042,2.357493,3.209376
7,melville-moby_dick.txt,ship,518,13,0.749020,0.750111,1.018835
8,melville-moby_dick.txt,captain,329,9,1.515910,1.519382,2.059037


### Declaración de uso de IA
Durante la realización de este trabajo se han utilizado herramientas de inteligencia artificial con el objetivo de comprender con mayor profundidad la materia y conceptos necesarios para realizar la actividad, mejorar el código redactado, explicar y realizar pseudocódigo, investigar acerca de librerías y redacción en Python.